In [5]:
from dotenv import load_dotenv
from langchain_core.output_parsers import PydanticOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field
from typing import Optional, Literal

In [2]:
load_dotenv()

True

In [3]:
model = ChatGoogleGenerativeAI(model='gemini-3.5-flash-lite',temperature=0.5)

In [10]:
Review_Text = '''Honestly blown away by the ping on this one. I was a bit skeptical buying a bat online without feeling the pickup in person first, but the balance is incredible—feels way lighter than the 1180g on the scale.

I knocked it in properly for about a week, took it straight into a weekend match, and the ball was just flying off the sweet spot with zero effort. The factory grip is a little thin, so I slapped an extra rubber grip on it, but for the performance you get out of this willow, it’s an absolute steal. Highly recommended for anyone who likes to hit through the covers! I would rate 4.5 out of 5. - Reviewed by Yash Raj'''

In [6]:
class ProductReview(BaseModel):
    Product : str = Field(description='Provide the name of the product')
    Pros : list[str] = Field(description='Provide the positive points of the product')
    Cons : list[str] = Field(description='Provide the negative points of the product',default="No info")
    Reviewer : Optional[str]= Field(description='Provide the reviewer name of the product',default="No Info")
    Ratings : float = Field(gt=1,lt=5,default=3,description='Return ratings of the reviewer for the current product')

In [7]:
jsonparser = PydanticOutputParser(pydantic_object=ProductReview)

In [8]:
template = PromptTemplate(template='Provide a Product Name, Pros, Cons, Ratings,Reviewer name for this specific Rating {ProductRating}  in {format}',
                          input_variables=['ProductRating'],partial_variables={'format':jsonparser.get_format_instructions()})

In [9]:
chains = template | model | jsonparser

In [11]:
respo = chains.invoke({'ProductRating':Review_Text})

In [17]:
respo.json()

C:\Users\yashr\AppData\Local\Temp\ipykernel_29448\1404799033.py:1: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  respo.json()


'{"Product":"Cricket Bat","Pros":["Incredible balance and feel (feels lighter than actual weight)","Great ping and sweet spot performance","Ball flies off the bat with zero effort","Excellent value for the performance (absolute steal)"],"Cons":["Factory grip is a little thin"],"Reviewer":"Yash Raj","Ratings":4.5,"Sentiments":"Positive"}'